# Python `str.replace()`: Master Substring Substitution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_replace_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.replace() Tutorial](https://www.plus2net.com/python/string-replace.php)**.

### Learning Objectives:
1. Understand substring replacement and string immutability in Python.
2. Master substitution limits using the optional `count` parameter.
3. Avoid common traps with chained replacements and character collisions.
4. Benchmark `str.replace()` against `re.sub()` and `str.translate()`.
5. Build real-world text pipelines: template rendering, PII masking, and path normalization.
6. Validate implementations with automated self-checking unit assertions.

## 1. Basic Substitution & Immutability Mechanics

`str.replace(old, new[, count])` searches for exact occurrences of `old` and replaces them with `new`.
Because Python strings are **immutable**, `replace()` always returns a new string and leaves the original untouched.

In [ ]:
# 1. Basic replacement
banner = "Welcome to PHP section of plus2net. PHP is versatile."
updated_banner = banner.replace("PHP", "Python")

print("Original: ", banner)
print("Updated:  ", updated_banner)

# 2. Verification of immutability
assert banner != updated_banner
assert banner == "Welcome to PHP section of plus2net. PHP is versatile."

# 3. Target absence: returns identical copy without error
assert banner.replace("Java", "Go") == banner

# 4. Case sensitivity: exact character match required
assert banner.replace("php", "Python") == banner

print("\nAll basic mechanics assertions passed!")

## 2. Controlling Replacements with the `count` Parameter

The `count` parameter specifies the maximum number of occurrences to substitute from left to right:
- `count = 1`: Replaces only the first match.
- `count = 0`: Performs zero replacements.
- `count < 0` (or omitted): Replaces every match across the string.

In [ ]:
log_line = "WARN: disk low; WARN: cpu high; WARN: ram full"

# Replace first match only
r1 = log_line.replace("WARN", "RESOLVED", 1)
print("Count = 1 :", r1)
assert r1.count("RESOLVED") == 1 and r1.count("WARN") == 2

# Replace first 2 matches
r2 = log_line.replace("WARN", "RESOLVED", 2)
print("Count = 2 :", r2)
assert r2.count("RESOLVED") == 2 and r2.count("WARN") == 1

# Count = 0
r0 = log_line.replace("WARN", "RESOLVED", 0)
print("Count = 0 :", r0)
assert r0 == log_line

# Count < 0 (all occurrences)
r_all = log_line.replace("WARN", "RESOLVED", -1)
print("Count = -1:", r_all)
assert r_all.count("RESOLVED") == 3 and r_all.count("WARN") == 0

print("\nCount limit assertions passed!")

## 3. Chained Replacements vs. The Multi-Mapping Collision Trap

When replacing multiple tokens, sequential `.replace().replace()` chains can cause unintended collisions if an earlier replacement produces tokens matching a subsequent rule.
For simultaneous multi-character substitutions or swapping tokens, use `str.translate()`.

In [ ]:
# Goal: Swap 'A' with 'B' and 'B' with 'A' in 'ABBA'
source = "ABBA"

# 1. Flawed sequential chaining
flawed = source.replace("A", "B").replace("B", "A")
print("Naive chain result:    ", flawed)  # Flawed: yields 'AAAA'
assert flawed == "AAAA"

# 2. Correct simultaneous swap via str.translate
table = str.maketrans({"A": "B", "B": "A"})
correct = source.translate(table)
print("Simultaneous swap result:", correct)  # Correct: yields 'BAAB'
assert correct == "BAAB"

# 3. Safe sequential chaining (when replacement sets do not intersect)
raw_html = "&lt;b&gt;Hello &amp; Welcome&lt;/b&gt;"
clean_text = raw_html.replace("&lt;", "<").replace("&gt;", ">").replace("&amp;", "&")
assert clean_text == "<b>Hello & Welcome</b>"
print("Safe unescape result:   ", clean_text)

## 4. Performance Comparison: `str.replace()` vs. `re.sub()` vs. `str.translate()`

Let's measure the throughput difference between standard literal replacement, regular expressions, and translation tables.

In [ ]:
import re
import time

test_corpus = "The quick brown fox jumps over the lazy dog. " * 1000
iterations = 500

# Benchmark str.replace()
t0 = time.perf_counter()
for _ in range(iterations):
    _ = test_corpus.replace("fox", "wolf")
t_replace = time.perf_counter() - t0

# Benchmark re.sub()
pattern = re.compile(r"fox")
t0 = time.perf_counter()
for _ in range(iterations):
    _ = pattern.sub("wolf", test_corpus)
t_regex = time.perf_counter() - t0

print(f"str.replace() execution time : {t_replace:.4f}s")
print(f"re.sub()      execution time : {t_regex:.4f}s")
print(f"str.replace() speedup factor : {t_regex / t_replace:.1f}x faster than re.sub()")

## 5. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.replace()`:
1. Dynamic Template Placeholder Rendering
2. Sensitive PII Redaction & Log Masking
3. Cross-Platform POSIX Path Normalization

In [ ]:
# Recipe 1: Template Placeholder Rendering
def render_template(template_str: str, context: dict) -> str:
    rendered = template_str
    for key, value in context.items():
        rendered = rendered.replace(f"{{{{{key}}}}}", str(value))
    return rendered

tmpl = "Hello {{user}}, your order #{{order_id}} has been shipped to {{city}}."
ctx = {"user": "Alice", "order_id": 84920, "city": "Seattle"}
out_msg = render_template(tmpl, ctx)
print("Recipe 1 Rendered:", out_msg)
assert out_msg == "Hello Alice, your order #84920 has been shipped to Seattle."

# Recipe 2: PII Redaction & Card Masking
def mask_credit_card(card_number: str) -> str:
    cleaned = card_number.replace("-", "").replace(" ", "")
    if len(cleaned) < 10:
        return card_number
    middle_len = len(cleaned) - 8
    masked_middle = "*" * middle_len
    unmasked_middle = cleaned[4:-4]
    return cleaned.replace(unmasked_middle, masked_middle, 1)

masked_card = mask_credit_card("4532-7589-1234-9876")
print("Recipe 2 Masked  :", masked_card)
assert masked_card == "4532********9876"

# Recipe 3: Path Normalization
def normalize_path(raw_path: str) -> str:
    posix_path = raw_path.replace("\\", "/")
    while "//" in posix_path:
        posix_path = posix_path.replace("//", "/")
    return posix_path

norm = normalize_path("C:\\Users\\Admin\\Documents\\reports\\q4.pdf")
print("Recipe 3 Path    :", norm)
assert norm == "C:/Users/Admin/Documents/reports/q4.pdf"

## 6. Production Implementation & Self-Checking Unit Assertions

This production-grade `StringSanitizer` class encapsulates clean currency parsing, count-bound substitutions, and safe template rendering with complete unit tests.

In [ ]:
class StringSanitizer:
    @staticmethod
    def clean_currency(raw_price: str) -> float:
        """Strips currency symbols and commas, returning float."""
        sanitized = raw_price.replace("$", "").replace("€", "").replace("£", "").replace(",", "").strip()
        return float(sanitized)

    @staticmethod
    def replace_first_n(text: str, old: str, new: str, limit: int) -> str:
        """Replaces at most 'limit' occurrences of 'old' with 'new'."""
        return text.replace(old, new, limit)

    @staticmethod
    def safe_template_render(template: str, mapping: dict) -> str:
        """Renders key-value placeholders safely without mutating inputs."""
        res = template
        for k, v in mapping.items():
            res = res.replace(f"${{{k}}}", str(v))
        return res


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
sanitizer = StringSanitizer()

# 1. Currency Cleaning
assert sanitizer.clean_currency("$1,499.50") == 1499.50
assert sanitizer.clean_currency("€ 250,000.00") == 250000.00
assert sanitizer.clean_currency("£12.99") == 12.99

# 2. Count Bound Replacements
sample = "cat-cat-cat-cat"
assert sanitizer.replace_first_n(sample, "cat", "dog", 1) == "dog-cat-cat-cat"
assert sanitizer.replace_first_n(sample, "cat", "dog", 2) == "dog-dog-cat-cat"
assert sanitizer.replace_first_n(sample, "cat", "dog", 0) == "cat-cat-cat-cat"
assert sanitizer.replace_first_n(sample, "cat", "dog", -1) == "dog-dog-dog-dog"

# 3. Template Rendering
tmpl = "Host: ${host}, Port: ${port}, SSL: ${ssl}"
context = {"host": "api.plus2net.com", "port": 443, "ssl": True}
rendered = sanitizer.safe_template_render(tmpl, context)
assert rendered == "Host: api.plus2net.com, Port: 443, SSL: True"

# 4. Target Absence & Immutability
original_str = "Python Basics"
assert original_str.replace("Java", "Kotlin") == "Python Basics"
assert original_str == "Python Basics"

# 5. Non-Overlapping Substitution
assert "xxxx".replace("xx", "y") == "yy"

# 6. Empty String Replacement Boundary
assert "abc".replace("", "-") == "-a-b-c-"

print("All str.replace() unit assertions passed successfully!")